In [7]:
# Check original files for duplicates 

import awswrangler as wr
import pandas as pd
import gc
import logging
logging.basicConfig(level=logging.INFO)

# S3 path to your specific parquet file/folder
volume_path = "s3://thesis--ec331-s3/enriched-volume-bids/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_202310010000_enriched/"

# Modified columns used to identify duplicates (removed BIDBAND as requested)
duplicate_keys = ["TRADINGDATE", "DUID", "PERIODID"]

print("\n--- LOADING SAMPLE OF RAISE1SEC DATA FOR DUPLICATE ANALYSIS ---")

# Discover the schema first without specifying columns
print(f"Attempting to discover schema from: {volume_path}")

try:
    # First, try to read a small sample with just a few rows to get the schema
    sample_df = wr.s3.read_parquet(
        path=volume_path,
        use_threads=True,
        chunked=10  # Just get a tiny sample to discover columns
    )
    sample_df = next(sample_df)  # Get the first chunk
    
    # Get the actual columns from the file
    available_columns = list(sample_df.columns)
    print(f"Discovered {len(available_columns)} available columns in the dataset")
    
    # Now read with a proper sample size
    print("\nReading data with all available columns...")
    volume_iter = wr.s3.read_parquet(
        path=volume_path,
        use_threads=True,
        chunked=200_000  # This will give you your first chunk of ~200K rows
    )
    
    # Only take the first chunk for analysis
    df_sample = next(volume_iter)
    print(f"Successfully loaded sample with {len(df_sample):,} rows")
    
    # Prepare data types
    if "TRADINGDATE" in df_sample.columns:
        df_sample["TRADINGDATE"] = pd.to_datetime(df_sample["TRADINGDATE"], errors="coerce")
    
    # Verify which columns we have for duplicate checking
    print("\nVerifying key columns for duplicate analysis:")
    for key in duplicate_keys:
        if key in df_sample.columns:
            print(f"✓ '{key}' is available")
        else:
            print(f"✗ '{key}' is missing")
    
    # If any key columns are missing, we need to adjust our duplicate keys
    if not all(key in df_sample.columns for key in duplicate_keys):
        available_keys = [key for key in duplicate_keys if key in df_sample.columns]
        print(f"\nWARNING: Not all duplicate keys are available in the dataset.")
        print(f"Adjusting duplicate keys from {duplicate_keys} to {available_keys}")
        duplicate_keys = available_keys
        
        if not duplicate_keys:
            raise ValueError("None of the specified duplicate keys are available in the dataset.")
    
    # --------------------------------------------
    # SIMPLIFIED DUPLICATE ANALYSIS
    # --------------------------------------------
    print(f"\n--- SIMPLIFIED DUPLICATE ANALYSIS BASED ON {duplicate_keys} ---")
    
    # 1. Find all duplicate key combinations
    dup_keys_mask = df_sample.duplicated(subset=duplicate_keys, keep=False)
    if not any(dup_keys_mask):
        print("No duplicates found in sample.")
    else:
        duplicate_rows = df_sample[dup_keys_mask].copy()
        print(f"Found {len(duplicate_rows):,} rows with duplicate keys in sample")
        
        # 2. Sort the data to group duplicate rows together
        sorted_dups = duplicate_rows.sort_values(by=duplicate_keys)
        
        # 3. Add a group identifier for each set of duplicates
        sorted_dups['dup_group'] = (
            sorted_dups[duplicate_keys].astype(str).agg('-'.join, axis=1)
        )
        
        # 4. Get statistics on duplicate groups
        dup_counts = sorted_dups.groupby('dup_group').size().reset_index(name='count')
        dup_counts = dup_counts.sort_values('count', ascending=False)
        
        print(f"\nFound {len(dup_counts):,} distinct duplicate groups")
        print(f"Maximum duplicates per key: {dup_counts['count'].max()}")

        # 5. Show all BIDBAND columns (if any exist)
        bidband_columns = [col for col in df_sample.columns if col.startswith('BIDBAND')]
        if bidband_columns:
            print(f"\nFound {len(bidband_columns)} BIDBAND columns: {bidband_columns}")
        else:
            print("\nNo BIDBAND columns found in the dataset")
            
        # 6. Show all BIDVOLUME columns (if any exist)
        bidvolume_columns = [col for col in df_sample.columns if col.startswith('BIDVOLUME')]
        if bidvolume_columns:
            print(f"\nFound {len(bidvolume_columns)} BIDVOLUME columns: {bidvolume_columns}")
        else:
            print("\nNo BIDVOLUME columns found in the dataset")
        
        # 7. Show simple statistics for a few duplicate groups
        print("\n--- SIMPLE STATISTICS FOR TOP DUPLICATE GROUPS ---")
        top_groups = dup_counts.head(3)['dup_group'].tolist()
        
        for i, group in enumerate(top_groups, 1):
            group_data = sorted_dups[sorted_dups['dup_group'] == group]
            print(f"\nDuplicate Group #{i}:")
            print(f"  - Contains {len(group_data)} rows with the same {duplicate_keys}")
            
            # Show the key values
            key_values = group_data[duplicate_keys].iloc[0].to_dict()
            for k, v in key_values.items():
                print(f"  - {k}: {v}")
        
        # 8. Provide summary of what to do
        print("\n--- DUPLICATE HANDLING RECOMMENDATION ---")
        print("""
        Based on this analysis, you should:
        
        1. Determine if these are true duplicates or if each row represents a different bid band
           (Check if there are BIDBAND1-BIDBAND10 columns that make each row unique)
        
        2. If they are true duplicates:
           - Use drop_duplicates() with appropriate subset and keep parameters
           - Example: df.drop_duplicates(subset=['TRADINGDATE', 'DUID', 'PERIODID'], keep='first')
        
        3. If they represent different bid bands:
           - This is expected behavior in power market bid data
           - When aggregating, use groupby() on the key columns and sum() the volumes
           - Example: df.groupby(['TRADINGDATE', 'DUID', 'PERIODID']).agg({
                       'BIDVOLUME1': 'sum', 'BIDVOLUME2': 'sum', ...
                     })
        """)
        
except Exception as e:
    print(f"Error during analysis: {str(e)}")

print("\n--- DUPLICATE ANALYSIS COMPLETE ---")

INFO:botocore.credentials:Found credentials from IAM Role: thesis



--- LOADING SAMPLE OF RAISE1SEC DATA FOR DUPLICATE ANALYSIS ---
Attempting to discover schema from: s3://thesis--ec331-s3/enriched-volume-bids/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_202310010000_enriched/


INFO:botocore.credentials:Found credentials from IAM Role: thesis


Discovered 47 available columns in the dataset

Reading data with all available columns...
Successfully loaded sample with 200,000 rows

Verifying key columns for duplicate analysis:
✓ 'TRADINGDATE' is available
✓ 'DUID' is available
✓ 'PERIODID' is available

--- SIMPLIFIED DUPLICATE ANALYSIS BASED ON ['TRADINGDATE', 'DUID', 'PERIODID'] ---
Found 199,999 rows with duplicate keys in sample

Found 1,445 distinct duplicate groups
Maximum duplicates per key: 289

No BIDBAND columns found in the dataset

No BIDVOLUME columns found in the dataset

--- SIMPLE STATISTICS FOR TOP DUPLICATE GROUPS ---

Duplicate Group #1:
  - Contains 289 rows with the same ['TRADINGDATE', 'DUID', 'PERIODID']
  - TRADINGDATE: 2023-10-22 00:00:00
  - DUID: DRVIOT02
  - PERIODID: 257.0

Duplicate Group #2:
  - Contains 289 rows with the same ['TRADINGDATE', 'DUID', 'PERIODID']
  - TRADINGDATE: 2023-10-22 00:00:00
  - DUID: DRVIOT02
  - PERIODID: 22.0

Duplicate Group #3:
  - Contains 289 rows with the same ['TRAD

In [ ]:
import awswrangler as wr
import pandas as pd
import gc
import logging
logging.basicConfig(level=logging.INFO)

# S3 path to your volume data
volume_path = "s3://thesis--ec331-s3/melted-volume-bids/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_202310010000_enriched/"

# Include all columns
volume_columns = [
    "I", 
    "BIDS", 
    "BIDOFFERPERIOD", 
    "1", 
    "DUID", 
    "BIDTYPE", 
    "TRADINGDATE", 
    "OFFERDATETIME", 
    "PERIODID", 
    "MAXAVAIL", 
    "FIXEDLOAD", 
    "RAMPUPRATE", 
    "RAMPDOWNRATE", 
    "ENABLEMENTMIN", 
    "ENABLEMENTMAX", 
    "LOWBREAKPOINT", 
    "HIGHBREAKPOINT", 
    "PASAAVAILABILITY", 
    "Participant", 
    "Station_Name", 
    "Region", 
    "Dispatch_Type", 
    "Category", 
    "Classification", 
    "Fuel_Source_-_Primary", 
    "Fuel_Source_-_Descriptor", 
    "Technology_Type_-_Primary", 
    "Technology_Type_-_Descriptor", 
    "Units", 
    "Aggregation", 
    "Reg_Cap_generation__MW_", 
    "Max_Cap_generation__MW_", 
    "Max_ROC/Min_generation", 
    "Reg_Cap_consumption__MW_", 
    "Max_Cap_consumption__MW_", 
    "Max_ROC/Min_consumption", 
    "Comments", 
    "BIDBAND", 
    "BIDVOLUME"
]

# Columns used to identify duplicates
duplicate_keys = ["TRADINGDATE", "DUID", "PERIODID", "BIDBAND"]

# IMPROVED APPROACH: Load a smaller sample for detailed duplicate analysis
print("\n--- LOADING SAMPLE OF VOLUME DATA FOR DUPLICATE ANALYSIS ---")

# Use the chunked parameter with a smaller first chunk
volume_iter = wr.s3.read_parquet(
    path=volume_path,
    path_suffix=".parquet",
    use_threads=True,
    columns=volume_columns,
    chunked=200_000  # This will give you your first chunk of ~200K rows
)

# Only take the first chunk for analysis
df_sample = next(volume_iter)
print(f"Loaded sample with {len(df_sample):,} rows")

# Prepare data types
if "TRADINGDATE" in df_sample.columns:
    df_sample["TRADINGDATE"] = pd.to_datetime(df_sample["TRADINGDATE"], errors="coerce")

# Only convert appropriate columns to category to avoid errors
cat_cols = ["DUID", "Region", "Dispatch_Type", "Category", "Classification", 
            "Fuel_Source_-_Primary", "Fuel_Source_-_Descriptor", 
            "Technology_Type_-_Primary", "Technology_Type_-_Descriptor"]
for col in cat_cols:
    if col in df_sample.columns:
        df_sample[col] = df_sample[col].astype("category")

# --------------------------------------------
# IMPROVED DUPLICATE ANALYSIS
# --------------------------------------------
print(f"\n--- DETAILED DUPLICATE ANALYSIS BASED ON {duplicate_keys} ---")

# 1. Find all duplicate key combinations
dup_keys_mask = df_sample.duplicated(subset=duplicate_keys, keep=False)
if not any(dup_keys_mask):
    print("No duplicates found in sample.")
else:
    duplicate_rows = df_sample[dup_keys_mask].copy()
    print(f"Found {len(duplicate_rows):,} rows with duplicate keys in sample")
    
    # 2. Sort the data to group duplicate rows together
    sorted_dups = duplicate_rows.sort_values(by=duplicate_keys)
    
    # 3. Add a group identifier for each set of duplicates
    sorted_dups['dup_group'] = (
        sorted_dups[duplicate_keys].astype(str).agg('-'.join, axis=1)
    )
    
    # 4. Get statistics on duplicate groups
    dup_counts = sorted_dups.groupby('dup_group').size().reset_index(name='count')
    dup_counts = dup_counts.sort_values('count', ascending=False)
    
    print(f"\nFound {len(dup_counts):,} distinct duplicate groups")
    print(f"Maximum duplicates per key: {dup_counts['count'].max()}")
    
    # 5. Display top duplicate groups
    num_groups_to_show = 3  # Reduced to 3 due to many columns
    top_groups = dup_counts.head(num_groups_to_show)['dup_group'].tolist()
    
    print(f"\n--- SHOWING FULL DETAILS FOR TOP {num_groups_to_show} DUPLICATE GROUPS ---")
    
    for i, group in enumerate(top_groups, 1):
        group_data = sorted_dups[sorted_dups['dup_group'] == group]
        print(f"\nDuplicate Group #{i} - {len(group_data)} occurrences:")
        
        # Show the common key values for this group
        key_values = group_data[duplicate_keys].iloc[0].to_dict()
        print("Key values:")
        for k, v in key_values.items():
            print(f"  {k}: {v}")
        
        # Check if all duplicate rows are EXACTLY identical
        identical_rows = group_data.iloc[:, :-1].duplicated().all()  # Exclude dup_group column
        print(f"Are all rows completely identical? {'Yes' if identical_rows else 'No'}")
        
        # Show all columns side by side for comparison (but just for a few rows if there are many)
        print("\nAll columns for comparison:")
        display_rows = min(5, len(group_data))  # Limit to 5 rows for readability
        
        # Focus on columns that might differ
        if not identical_rows:
            print("Columns with differences:")
            diff_cols = []
            all_cols = group_data.columns.tolist()
            all_cols.remove('dup_group')  # Remove the added column
            
            for col in all_cols:
                if group_data[col].nunique() > 1:
                    diff_cols.append(col)
                    print(f"  - {col} has {group_data[col].nunique()} different values")
            
            # Show a more focused view with just key columns and columns that differ
            columns_to_show = duplicate_keys + diff_cols
            columns_to_show = list(dict.fromkeys(columns_to_show))  # Remove duplicates while preserving order
            
            with pd.option_context('display.max_rows', None, 
                                  'display.max_columns', None,
                                  'display.width', 1000):
                print("\nFocused view - showing only key columns and columns with differences:")
                print(group_data[columns_to_show].head(display_rows))
        
        # If all rows are identical or there are few columns with differences, show all columns
        with pd.option_context('display.max_rows', None, 
                              'display.max_columns', None,
                              'display.width', 1000):
            print("\nFull view (all columns):")
            pd.set_option('display.max_colwidth', 20)  # Limit column width for better display
            print(group_data.head(display_rows))
    
    # 6. Focused comparison of pairs
    print("\n--- PAIR-WISE COMPARISON OF DUPLICATES ---")
    # Find groups with exactly 2 rows for clear comparison
    dup_groups_with_two = dup_counts[dup_counts['count'] == 2]['dup_group'].tolist()
    if dup_groups_with_two:
        sample_size = min(3, len(dup_groups_with_two))
        sample_groups = dup_groups_with_two[:sample_size]
        
        for i, group in enumerate(sample_groups, 1):
            print(f"\nDuplicate Pair Example #{i}:")
            pair = sorted_dups[sorted_dups['dup_group'] == group].reset_index(drop=True)
            
            # Compare values across rows
            if len(pair) == 2:
                diff_cols = []
                print("\nColumn differences:")
                for col in pair.columns:
                    if col != 'dup_group' and pair[col][0] != pair[col][1]:
                        diff_cols.append(col)
                        print(f"  - {col}: {pair[col][0]} vs {pair[col][1]}")
                
                if not diff_cols:
                    print("  All columns are identical!")
                else:
                    # Show just the columns that differ
                    columns_to_show = duplicate_keys + diff_cols
                    columns_to_show = list(dict.fromkeys(columns_to_show))  # Remove duplicates
                    print("\nFocused view of differences:")
                    pd.set_option('display.max_columns', None)
                    print(pair[columns_to_show])
    
    # 7. Summary statistics on what differs between duplicates
    print("\n--- SUMMARY OF DIFFERENCES BETWEEN DUPLICATES ---")
    
    # Initialize counters
    total_dup_groups = len(dup_counts)
    identical_groups = 0
    diff_col_counts = {}
    
    # Analyze a sample of duplicate groups
    sample_size = min(100, total_dup_groups)
    sample_groups = dup_counts['dup_group'].tolist()[:sample_size]
    
    for group in sample_groups:
        group_data = sorted_dups[sorted_dups['dup_group'] == group]
        
        # Check if all rows in this group are identical
        if group_data.iloc[:, :-1].duplicated().all():  # Exclude dup_group column
            identical_groups += 1
        else:
            # Find which columns differ
            for col in group_data.columns:
                if col != 'dup_group' and group_data[col].nunique() > 1:
                    diff_col_counts[col] = diff_col_counts.get(col, 0) + 1
    
    # Report results
    print(f"Analyzed {sample_size} of {total_dup_groups} duplicate groups")
    print(f"Completely identical groups: {identical_groups} ({identical_groups/sample_size*100:.1f}%)")
    
    if diff_col_counts:
        print("\nColumns most likely to differ between duplicates:")
        for col, count in sorted(diff_col_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
            print(f"  - {col}: differs in {count} groups ({count/sample_size*100:.1f}%)")
    
    # 8. Provide a summary of what to do next
    print("\n--- DUPLICATE HANDLING RECOMMENDATIONS ---")
    print("Based on the analysis, you might want to:")
    print("1. Keep the first occurrence of each key combination")
    print("2. Aggregate data (e.g., sum, mean) for each key combination")
    print("3. Investigate why duplicates exist - could be a data quality issue")
    
    # 9. Show example code for handling duplicates
    print("\nExample code to handle duplicates:")
    print("""
    # To keep only the first occurrence:
    df_deduplicated = df.drop_duplicates(subset=duplicate_keys, keep='first')
    
    # To aggregate (if appropriate):
    df_aggregated = df.groupby(duplicate_keys).agg({
        'BIDVOLUME': 'mean',  # or 'sum', 'min', 'max', etc.
        # other numeric columns...
    }).reset_index()
    """)

print("\n--- DUPLICATE ANALYSIS COMPLETE ---")

In [ ]:
import awswrangler as wr
import pandas as pd
import gc
import logging

logging.basicConfig(level=logging.INFO)

# S3 path to your volume data
volume_path = "s3://thesis--ec331-s3/melted-volume-bids/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_202310010000_enriched/"

# Columns to load (adjust as needed)
volume_columns = [
    "TRADINGDATE",
    "DUID",
    "PERIODID",
    "BIDBAND",
    "BIDVOLUME",
    # ... any other columns you want ...
]

# Columns used to identify duplicates
duplicate_keys = ["TRADINGDATE", "DUID", "PERIODID", "BIDBAND"]

print("\n--- LOADING VOLUME DATA IN CHUNKS ---")
volume_iter = wr.s3.read_parquet(
    path=volume_path,
    path_suffix=".parquet",
    use_threads=True,
    columns=volume_columns,
    chunked=50_000
)

df_list = []
total_rows = 0

for i, chunk in enumerate(volume_iter, start=1):
    print(f"Reading volume chunk {i}...")

    # Convert TRADINGDATE to datetime if needed
    if "TRADINGDATE" in chunk.columns:
        chunk["TRADINGDATE"] = pd.to_datetime(chunk["TRADINGDATE"], errors="coerce")

    # Convert relevant columns to category to reduce memory usage
    cat_cols = ["DUID", "PERIODID", "BIDBAND"]
    for col in cat_cols:
        if col in chunk.columns:
            chunk[col] = chunk[col].astype("category")

    # Downcast numeric columns if desired
    if "BIDVOLUME" in chunk.columns:
        chunk["BIDVOLUME"] = chunk["BIDVOLUME"].astype("float32")

    chunk_len = len(chunk)
    total_rows += chunk_len
    df_list.append(chunk)
    print(f"  -> Chunk {i} has {chunk_len:,} rows")

    gc.collect()

if not df_list:
    print("No volume data found. Exiting.")
else:
    # Combine all chunks into a single DataFrame
    df = pd.concat(df_list, ignore_index=True)
    print(f"\nTotal rows combined: {total_rows:,}")

    # Clean up
    del df_list
    gc.collect()

    # --------------------------------------------
    # Check for duplicate key combinations
    # --------------------------------------------
    print(f"\n--- CHECKING FOR DUPLICATES BASED ON {duplicate_keys} ---")

    # Mark rows as duplicates if they share the same subset of keys
    duplicate_mask = df.duplicated(subset=duplicate_keys, keep=False)
    duplicates = df[duplicate_mask]

    if duplicates.empty:
        print(f"No duplicates found for {duplicate_keys}.")
    else:
        print(f"Duplicates found for {duplicate_keys}.")

        # Print how many duplicates
        print(f"Total duplicate rows: {len(duplicates):,}")

        # Show a small sample of the duplicate rows (e.g., first 20)
        sample_size = min(20, len(duplicates))
        print(f"\n--- SAMPLE OF {sample_size} DUPLICATE ROWS ---")
        with pd.option_context('display.max_rows', None,
                               'display.max_columns', None,
                               'display.max_colwidth', None):
            display(duplicates.head(sample_size))

    # Optionally, show a condensed summary of how many times each key appears
    dup_counts = (
        df.groupby(duplicate_keys)
          .size()
          .reset_index(name="count")
    )
    # Filter to combos that appear more than once
    duplicate_groups = dup_counts[dup_counts["count"] > 1]

    if not duplicate_groups.empty:
        print("\nSummary of key combos that appear more than once:")
        # Show only the first few lines
        display(duplicate_groups.head(20))
    else:
        print("\nAll key combos appear only once.")

    print("\n--- DUPLICATE CHECK COMPLETE ---")

In [ ]:
grouped = duplicates.groupby(["TRADINGDATE", "DUID", "PERIODID", "BIDBAND"])
for keys, group_df in grouped:
    if len(group_df) > 1:
        print(f"--- DUPLICATES FOR {keys} ---")
        display(group_df)

In [ ]:
import awswrangler as wr
import pandas as pd
import gc
import logging

logging.basicConfig(level=logging.INFO)

# Define the S3 path to read from (the output folder of your last script)
price_path = "s3://thesis--ec331-s3/de-duped-price-bids/"

# Columns to load (adjust as needed)
price_columns = [
    "I", 
    "BIDS", 
    "BIDDAYOFFER", 
    "1", 
    "DUID", 
    "BIDTYPE", 
    "SETTLEMENTDATE", 
    "OFFERDATE", 
    "VERSIONNO", 
    "PARTICIPANTID", 
    "DAILYENERGYCONSTRAINT", 
    "REBIDEXPLANATION", 
    "MINIMUMLOAD", 
    "T1", 
    "T2", 
    "T3", 
    "T4", 
    "NORMALSTATUS", 
    "LASTCHANGED", 
    "ENTRYTYPE", 
    "REBID_EVENT_TIME", 
    "REBID_AWARE_TIME", 
    "REBID_DECISION_TIME", 
    "REBID_CATEGORY", 
    "REFERENCE_ID", 
    "BIDBAND", 
    "BIDPRICE"
]

print("\n--- LOADING PRICE DATA IN CHUNKS ---")
price_iter = wr.s3.read_parquet(
    path=price_path,
    path_suffix=".parquet",
    use_threads=True,
    columns=price_columns,
    chunked=50_000
)

price_df_list = []
total_price_rows = 0

for i, price_chunk in enumerate(price_iter, start=1):
    print(f"Reading price chunk {i}...")

    # Convert SETTLEMENTDATE to datetime
    price_chunk["SETTLEMENTDATE"] = pd.to_datetime(price_chunk["SETTLEMENTDATE"], errors="coerce")

    # Convert relevant columns to category (optional)
    cat_cols_price = [
        "DUID", "BIDTYPE", "PARTICIPANTID", "BIDBAND", 
        "I", "BIDS", "BIDDAYOFFER", "OFFERDATE", "VERSIONNO", 
        "NORMALSTATUS", "ENTRYTYPE", "REBID_CATEGORY", "REFERENCE_ID"
    ]
    for col in cat_cols_price:
        if col in price_chunk.columns:
            price_chunk[col] = price_chunk[col].astype("category")

    # Downcast BIDPRICE to float32
    if "BIDPRICE" in price_chunk.columns:
        price_chunk["BIDPRICE"] = price_chunk["BIDPRICE"].astype("float32")

    chunk_len = len(price_chunk)
    total_price_rows += chunk_len
    price_df_list.append(price_chunk)
    print(f"  -> Chunk {i} has {chunk_len:,} rows")

    gc.collect()

if not price_df_list:
    print("No price data found. Exiting.")
else:
    # Combine all chunks into a single DataFrame
    price_df = pd.concat(price_df_list, ignore_index=True)
    print(f"\nTotal PRICE rows combined: {total_price_rows:,}")

    # Clean up
    del price_df_list
    gc.collect()

    # --------------------------------------------
    # Check for duplicate key combinations
    # --------------------------------------------
    print("\n--- CHECKING FOR DUPLICATES BASED ON [SETTLEMENTDATE, DUID, BIDBAND] ---")

    duplicate_mask = price_df.duplicated(subset=["SETTLEMENTDATE", "DUID", "BIDBAND"], keep=False)
    duplicates = price_df[duplicate_mask]

    if duplicates.empty:
        print("No duplicates found for [SETTLEMENTDATE, DUID, BIDBAND].")
    else:
        print("Duplicates found for [SETTLEMENTDATE, DUID, BIDBAND].")

        # Print how many duplicates
        print(f"Total duplicate rows: {len(duplicates):,}")

        # Show a small sample of the duplicate rows (e.g., first 20)
        sample_size = min(20, len(duplicates))
        print(f"\n--- SAMPLE OF {sample_size} DUPLICATE ROWS ---")
        with pd.option_context('display.max_rows', None,
                               'display.max_columns', None,
                               'display.max_colwidth', None):
            display(duplicates.head(sample_size))

    # Optionally, show a condensed summary of how many times each key appears
    dup_counts = (
        price_df.groupby(["SETTLEMENTDATE", "DUID", "BIDBAND"])
                .size()
                .reset_index(name="count")
    )
    # Filter to combos that appear more than once
    duplicate_groups = dup_counts[dup_counts["count"] > 1]

    if not duplicate_groups.empty:
        print("\nSummary of key combos that appear more than once:")
        # Show only the first few lines
        display(duplicate_groups.head(20))
    else:
        print("\nAll key combos appear only once.")

    print("\n--- DUPLICATE CHECK COMPLETE ---")

In [ ]:
import boto3
import pandas as pd
import pyarrow.parquet as pq
from io import BytesIO
import random
from typing import List, Dict, Any, Tuple
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def list_parquet_files(bucket_name: str, prefix: str, max_files: int = 100) -> List[str]:
    """
    List parquet files in the specified S3 bucket and prefix.
    
    Args:
        bucket_name: Name of the S3 bucket
        prefix: Prefix/folder to search in
        max_files: Maximum number of files to return
    
    Returns:
        List of S3 keys for parquet files
    """
    s3_client = boto3.client('s3')
    
    try:
        paginator = s3_client.get_paginator('list_objects_v2')
        pages = paginator.paginate(Bucket=bucket_name, Prefix=prefix)
        
        file_keys = []
        for page in pages:
            if 'Contents' not in page:
                continue
                
            for obj in page['Contents']:
                if obj['Key'].endswith('.parquet'):
                    file_keys.append(obj['Key'])
                    if len(file_keys) >= max_files:
                        return file_keys
        
        return file_keys
    except Exception as e:
        logger.error(f"Error listing parquet files: {e}")
        raise

def sample_parquet_files(bucket_name: str, prefix: str, sample_size: int = 5) -> List[str]:
    """
    Sample a random subset of parquet files from the S3 bucket.
    
    Args:
        bucket_name: Name of the S3 bucket
        prefix: Prefix/folder to search in
        sample_size: Number of files to sample
    
    Returns:
        List of sampled file keys
    """
    all_files = list_parquet_files(bucket_name, prefix)
    logger.info(f"Found {len(all_files)} parquet files")
    
    if not all_files:
        logger.warning("No parquet files found!")
        return []
    
    if len(all_files) <= sample_size:
        return all_files
    
    return random.sample(all_files, sample_size)

def read_parquet_from_s3(bucket_name: str, key: str) -> pd.DataFrame:
    """
    Read a parquet file from S3 into a pandas DataFrame.
    
    Args:
        bucket_name: Name of the S3 bucket
        key: S3 key for the parquet file
        
    Returns:
        DataFrame with the parquet data
    """
    s3_client = boto3.client('s3')
    
    try:
        logger.info(f"Reading file s3://{bucket_name}/{key}")
        response = s3_client.get_object(Bucket=bucket_name, Key=key)
        buffer = BytesIO(response['Body'].read())
        
        # For large files, we can use pyarrow to read only a sample
        # This is more efficient than loading the entire file into memory
        table = pq.read_table(buffer)
        df = table.to_pandas()
        
        logger.info(f"Successfully read {len(df)} rows")
        return df
    except Exception as e:
        logger.error(f"Error reading parquet file s3://{bucket_name}/{key}: {e}")
        raise

def check_duplicates(df: pd.DataFrame) -> Tuple[bool, pd.DataFrame, Dict[str, Any]]:
    """
    Check for duplicate rows in a DataFrame and return statistics.
    
    Args:
        df: DataFrame to check for duplicates
        
    Returns:
        Tuple containing:
        - Boolean indicating if duplicates were found
        - DataFrame with only duplicate rows
        - Dictionary with duplicate statistics
    """
    # Count duplicated rows
    duplicated = df.duplicated(keep='first')
    dup_count = duplicated.sum()
    
    # Create a DataFrame with only the duplicated rows
    duplicates_df = df[df.duplicated(keep=False)].sort_values(by=df.columns.tolist())
    
    # Calculate statistics
    stats = {
        'total_rows': len(df),
        'duplicate_rows': dup_count,
        'unique_rows': len(df) - dup_count,
        'duplicate_percentage': (dup_count / len(df) * 100) if len(df) > 0 else 0,
    }
    
    has_duplicates = dup_count > 0
    
    return has_duplicates, duplicates_df, stats

def analyze_s3_parquet_duplicates(s3_uri: str, sample_size: int = 5, unique_key_columns: List[str] = None) -> None:
    """
    Main function to analyze duplicates in parquet files stored in S3.
    
    Args:
        s3_uri: S3 URI in the format 's3://bucket-name/prefix'
        sample_size: Number of files to sample
        unique_key_columns: Optional list of columns that should form a unique key
    """
    # Parse S3 URI
    parts = s3_uri.replace('s3://', '').split('/', 1)
    if len(parts) != 2:
        raise ValueError(f"Invalid S3 URI format: {s3_uri}. Expected format: s3://bucket-name/prefix")
    
    bucket_name, prefix = parts
    
    logger.info(f"Analyzing parquet files in s3://{bucket_name}/{prefix}")
    
    # Sample files
    sampled_files = sample_parquet_files(bucket_name, prefix, sample_size)
    
    if not sampled_files:
        logger.warning("No parquet files found to analyze")
        return
    
    # Process each file
    all_results = []
    
    for file_key in sampled_files:
        try:
            df = read_parquet_from_s3(bucket_name, file_key)
            
            # Check for duplicates in the default way (all columns)
            has_duplicates, duplicates_df, stats = check_duplicates(df)
            
            # If specific key columns were provided, also check those
            if unique_key_columns:
                # Make sure all columns exist
                missing_cols = [col for col in unique_key_columns if col not in df.columns]
                if missing_cols:
                    logger.warning(f"Missing columns in the dataset: {missing_cols}")
                    valid_key_columns = [col for col in unique_key_columns if col in df.columns]
                else:
                    valid_key_columns = unique_key_columns
                
                if valid_key_columns:
                    # Check for duplicates based on the specified columns
                    key_duplicated = df.duplicated(subset=valid_key_columns, keep='first')
                    key_dup_count = key_duplicated.sum()
                    key_duplicates_df = df[df.duplicated(subset=valid_key_columns, keep=False)].sort_values(
                        by=valid_key_columns)
                    
                    key_stats = {
                        'key_columns': valid_key_columns,
                        'duplicate_keys': key_dup_count,
                        'duplicate_key_percentage': (key_dup_count / len(df) * 100) if len(df) > 0 else 0,
                    }
                    
                    stats.update(key_stats)
                    has_key_duplicates = key_dup_count > 0
                    
                    if has_key_duplicates:
                        logger.warning(f"Found {key_dup_count} duplicates based on key columns {valid_key_columns}")
                        # Display a sample of duplicates
                        if not key_duplicates_df.empty and len(key_duplicates_df) > 0:
                            sample_size = min(5, len(key_duplicates_df))
                            logger.info(f"Sample of duplicates based on key columns:\n{key_duplicates_df.head(sample_size)}")
            
            # Save results
            result = {
                'file_key': file_key,
                'has_duplicates': has_duplicates,
                'stats': stats
            }
            all_results.append(result)
            
            # Log findings
            if has_duplicates:
                logger.warning(f"Found {stats['duplicate_rows']} duplicates in {file_key} ({stats['duplicate_percentage']:.2f}%)")
                # Display a sample of duplicates
                if not duplicates_df.empty and len(duplicates_df) > 0:
                    sample_size = min(5, len(duplicates_df))
                    logger.info(f"Sample of duplicates:\n{duplicates_df.head(sample_size)}")
            else:
                logger.info(f"No duplicates found in {file_key}")
                
        except Exception as e:
            logger.error(f"Error processing file {file_key}: {e}")
            all_results.append({
                'file_key': file_key,
                'error': str(e)
            })
    
    # Summarize findings
    logger.info("\n===== SUMMARY =====")
    files_with_duplicates = sum(1 for r in all_results if r.get('has_duplicates', False))
    logger.info(f"Processed {len(sampled_files)} parquet files")
    logger.info(f"Files with duplicates: {files_with_duplicates}")
    logger.info(f"Files without duplicates: {len(sampled_files) - files_with_duplicates}")
    
    # Calculate average duplicate percentage
    duplicate_percentages = [r['stats']['duplicate_percentage'] 
                            for r in all_results 
                            if 'stats' in r and 'duplicate_percentage' in r['stats']]
    
    if duplicate_percentages:
        avg_duplicate_percentage = sum(duplicate_percentages) / len(duplicate_percentages)
        logger.info(f"Average duplicate percentage: {avg_duplicate_percentage:.2f}%")
    
    return all_results

if __name__ == "__main__":
    # Example usage
    s3_uri = "s3://thesis--ec331-s3/melted-volume-bids/RAISE1SEC_PUBLIC_DVD_BIDPEROFFER2_202310010000_enriched/"
    
    # You may want to specify columns that should form a unique key
    # These would be columns that should uniquely identify a row
    # For example: ['timestamp', 'security_id', 'order_id']
    unique_key_columns = None  # Replace with actual column names if known
    
    results = analyze_s3_parquet_duplicates(s3_uri, sample_size=3, unique_key_columns=unique_key_columns)

In [ ]:
import boto3
import pandas as pd
import pyarrow.parquet as pq
from io import BytesIO
import random
from typing import List, Dict, Any, Tuple
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def list_parquet_files(bucket_name: str, prefix: str, max_files: int = 100) -> List[str]:
    """
    List parquet files in the specified S3 bucket and prefix.
    
    Args:
        bucket_name: Name of the S3 bucket
        prefix: Prefix/folder to search in
        max_files: Maximum number of files to return
    
    Returns:
        List of S3 keys for parquet files
    """
    s3_client = boto3.client('s3')
    
    try:
        paginator = s3_client.get_paginator('list_objects_v2')
        pages = paginator.paginate(Bucket=bucket_name, Prefix=prefix)
        
        file_keys = []
        for page in pages:
            if 'Contents' not in page:
                continue
                
            for obj in page['Contents']:
                if obj['Key'].endswith('.parquet'):
                    file_keys.append(obj['Key'])
                    if len(file_keys) >= max_files:
                        return file_keys
        
        return file_keys
    except Exception as e:
        logger.error(f"Error listing parquet files: {e}")
        raise

def sample_parquet_files(bucket_name: str, prefix: str, sample_size: int = 5) -> List[str]:
    """
    Sample a random subset of parquet files from the S3 bucket.
    
    Args:
        bucket_name: Name of the S3 bucket
        prefix: Prefix/folder to search in
        sample_size: Number of files to sample
    
    Returns:
        List of sampled file keys
    """
    all_files = list_parquet_files(bucket_name, prefix)
    logger.info(f"Found {len(all_files)} parquet files")
    
    if not all_files:
        logger.warning("No parquet files found!")
        return []
    
    if len(all_files) <= sample_size:
        return all_files
    
    return random.sample(all_files, sample_size)

def read_parquet_from_s3(bucket_name: str, key: str) -> pd.DataFrame:
    """
    Read a parquet file from S3 into a pandas DataFrame.
    
    Args:
        bucket_name: Name of the S3 bucket
        key: S3 key for the parquet file
        
    Returns:
        DataFrame with the parquet data
    """
    s3_client = boto3.client('s3')
    
    try:
        logger.info(f"Reading file s3://{bucket_name}/{key}")
        response = s3_client.get_object(Bucket=bucket_name, Key=key)
        buffer = BytesIO(response['Body'].read())
        
        # For large files, we can use pyarrow to read only a sample
        # This is more efficient than loading the entire file into memory
        table = pq.read_table(buffer)
        df = table.to_pandas()
        
        logger.info(f"Successfully read {len(df)} rows")
        return df
    except Exception as e:
        logger.error(f"Error reading parquet file s3://{bucket_name}/{key}: {e}")
        raise

def check_duplicates(df: pd.DataFrame) -> Tuple[bool, pd.DataFrame, Dict[str, Any]]:
    """
    Check for duplicate rows in a DataFrame and return statistics.
    
    Args:
        df: DataFrame to check for duplicates
        
    Returns:
        Tuple containing:
        - Boolean indicating if duplicates were found
        - DataFrame with only duplicate rows
        - Dictionary with duplicate statistics
    """
    # Count duplicated rows
    duplicated = df.duplicated(keep='first')
    dup_count = duplicated.sum()
    
    # Create a DataFrame with only the duplicated rows
    duplicates_df = df[df.duplicated(keep=False)].sort_values(by=df.columns.tolist())
    
    # Calculate statistics
    stats = {
        'total_rows': len(df),
        'duplicate_rows': dup_count,
        'unique_rows': len(df) - dup_count,
        'duplicate_percentage': (dup_count / len(df) * 100) if len(df) > 0 else 0,
    }
    
    has_duplicates = dup_count > 0
    
    return has_duplicates, duplicates_df, stats

def analyze_s3_parquet_duplicates(s3_uri: str, sample_size: int = 5, unique_key_columns: List[str] = None) -> None:
    """
    Main function to analyze duplicates in parquet files stored in S3.
    
    Args:
        s3_uri: S3 URI in the format 's3://bucket-name/prefix'
        sample_size: Number of files to sample
        unique_key_columns: Optional list of columns that should form a unique key
    """
    # Parse S3 URI
    parts = s3_uri.replace('s3://', '').split('/', 1)
    if len(parts) != 2:
        raise ValueError(f"Invalid S3 URI format: {s3_uri}. Expected format: s3://bucket-name/prefix")
    
    bucket_name, prefix = parts
    
    logger.info(f"Analyzing parquet files in s3://{bucket_name}/{prefix}")
    
    # Sample files
    sampled_files = sample_parquet_files(bucket_name, prefix, sample_size)
    
    if not sampled_files:
        logger.warning("No parquet files found to analyze")
        return
    
    # Process each file
    all_results = []
    
    for file_key in sampled_files:
        try:
            df = read_parquet_from_s3(bucket_name, file_key)
            
            # Check for duplicates in the default way (all columns)
            has_duplicates, duplicates_df, stats = check_duplicates(df)
            
            # If specific key columns were provided, also check those
            if unique_key_columns:
                # Make sure all columns exist
                missing_cols = [col for col in unique_key_columns if col not in df.columns]
                if missing_cols:
                    logger.warning(f"Missing columns in the dataset: {missing_cols}")
                    valid_key_columns = [col for col in unique_key_columns if col in df.columns]
                else:
                    valid_key_columns = unique_key_columns
                
                if valid_key_columns:
                    # Check for duplicates based on the specified columns
                    key_duplicated = df.duplicated(subset=valid_key_columns, keep='first')
                    key_dup_count = key_duplicated.sum()
                    key_duplicates_df = df[df.duplicated(subset=valid_key_columns, keep=False)].sort_values(
                        by=valid_key_columns)
                    
                    key_stats = {
                        'key_columns': valid_key_columns,
                        'duplicate_keys': key_dup_count,
                        'duplicate_key_percentage': (key_dup_count / len(df) * 100) if len(df) > 0 else 0,
                    }
                    
                    stats.update(key_stats)
                    has_key_duplicates = key_dup_count > 0
                    
                    if has_key_duplicates:
                        logger.warning(f"Found {key_dup_count} duplicates based on key columns {valid_key_columns}")
                        # Display a sample of duplicates
                        if not key_duplicates_df.empty and len(key_duplicates_df) > 0:
                            sample_size = min(5, len(key_duplicates_df))
                            logger.info(f"Sample of duplicates based on key columns:\n{key_duplicates_df.head(sample_size)}")
            
            # Save results
            result = {
                'file_key': file_key,
                'has_duplicates': has_duplicates,
                'stats': stats
            }
            all_results.append(result)
            
            # Log findings
            if has_duplicates:
                logger.warning(f"Found {stats['duplicate_rows']} duplicates in {file_key} ({stats['duplicate_percentage']:.2f}%)")
                # Display a sample of duplicates
                if not duplicates_df.empty and len(duplicates_df) > 0:
                    sample_size = min(5, len(duplicates_df))
                    logger.info(f"Sample of duplicates:\n{duplicates_df.head(sample_size)}")
            else:
                logger.info(f"No duplicates found in {file_key}")
                
        except Exception as e:
            logger.error(f"Error processing file {file_key}: {e}")
            all_results.append({
                'file_key': file_key,
                'error': str(e)
            })
    
    # Summarize findings
    logger.info("\n===== SUMMARY =====")
    files_with_duplicates = sum(1 for r in all_results if r.get('has_duplicates', False))
    logger.info(f"Processed {len(sampled_files)} parquet files")
    logger.info(f"Files with duplicates: {files_with_duplicates}")
    logger.info(f"Files without duplicates: {len(sampled_files) - files_with_duplicates}")
    
    # Calculate average duplicate percentage
    duplicate_percentages = [r['stats']['duplicate_percentage'] 
                            for r in all_results 
                            if 'stats' in r and 'duplicate_percentage' in r['stats']]
    
    if duplicate_percentages:
        avg_duplicate_percentage = sum(duplicate_percentages) / len(duplicate_percentages)
        logger.info(f"Average duplicate percentage: {avg_duplicate_percentage:.2f}%")
    
    return all_results

if __name__ == "__main__":
    # Example usage
    s3_uri = "s3://thesis--ec331-s3/merged-price-volume-bids/20231020_to_20231031/"
    
    # You may want to specify columns that should form a unique key
    # These would be columns that should uniquely identify a row
    # For example: ['timestamp', 'security_id', 'order_id']
    unique_key_columns = None  # Replace with actual column names if known
    
    results = analyze_s3_parquet_duplicates(s3_uri, sample_size=3, unique_key_columns=unique_key_columns)